In [216]:
import pypsa
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import tz_pypsa.plotting as pypsa_plotting

import plotly

In [217]:
# Load or create a PyPSA network
network = pypsa.Network()

# Import the .nc file into PyPSA (e.g. tz-apg-v1_scenario-bau.nc)
network.import_from_netcdf(path = '/Users/yantidiah/Documents/CFE/runs/MYSPE_fullCFE_18May_TP2_newcost/solved_networks/brownfield_2030.nc')

network

INFO:pypsa.io:Imported network brownfield_2030.nc has buses, carriers, generators, links, loads, storage_units


PyPSA Network
Components:
 - Bus: 10
 - Carrier: 55
 - Generator: 100
 - Link: 19
 - Load: 9
 - StorageUnit: 16
Snapshots: 8760

In [219]:
network.loads_t.p_set.sum()

Load
IDNKA             2.063000e+07
MYSPE             2.026660e+08
MYSPE C&I Load    1.348801e+07
MYSSH             7.531000e+06
MYSSK             6.358300e+07
SGPXX             8.914300e+07
THACE             1.658125e+08
THANO             5.614598e+07
THASO             3.052139e+07
dtype: float64

In [230]:
network.loads_t.p_set["MYSPE"].loc['2030-01-01 00:00:00':'2030-12-31 23:00:00']

snapshot
2030-01-01 00:00:00    16329.913509
2030-01-01 01:00:00    16903.478529
2030-01-01 02:00:00    17588.570643
2030-01-01 03:00:00    17778.245743
2030-01-01 04:00:00    17661.408800
                           ...     
2030-12-31 19:00:00    18297.942142
2030-12-31 20:00:00    17690.227904
2030-12-31 21:00:00    17288.888443
2030-12-31 22:00:00    17411.035235
2030-12-31 23:00:00    16656.144954
Name: MYSPE, Length: 8760, dtype: float64

In [ ]:
network.statistics(groupby=['bus','carrier']).loc['Generator']

In [ ]:
network.statistics.installed_capacity(groupby=['bus','carrier']).loc['Generator']

In [ ]:
network.statistics.revenue(groupby=['bus','carrier']).loc['Generator']

In [ ]:
fig = pypsa_plotting.energy_balance(network, 2030, show_imports=True, nodes_to_plot=['MYSPE','MYSSK','MYSSH','SGPXX'])
fig.show()
# fig.write_html('energy_balance.html')

In [ ]:
fig = pypsa_plotting.total_capacity(network, 2030, nodes_to_plot=['MYSPE','MYSSK','MYSSH','SGPXX'])
fig.show()
# fig.write_html('total_capacity.html')

In [ ]:
import plotly.express as px

nodes_to_plot=['MYSPE','MYSSK','MYSSH','SGPXX']

imports = (
    network
    .links_t
    .p0
    .resample('YE')
    .sum()
    .div(1e6)
    .melt()
)
imports = imports.loc[(imports['Link'].str.contains('|'.join(nodes_to_plot))) &
                      ~(imports['Link'].str.contains('C&I'))]
imports['names'] = imports['Link'].replace('-ext-2030', '', regex=True)

fig = px.bar(imports, x='names', y='value')

fig.update_layout(
    yaxis_title='TWh',
    xaxis_title='',
    title='Traded electricity',
    width=1200,
    height=500,
    xaxis_tickangle=-45
)
fig.show()

# fig.write_html('traded_electricity.html')

In [ ]:
network.loads

In [236]:
## DISPATCH AT THE HIGHEST DEMAND WEEK

resample = 'H'
iso_code = 'MYSPE'

# get generation
generation = (
    network
    .generators_t
    .p
    .loc['2030-05-07 00:00:00':'2030-05-14 23:00:00']
    .filter(regex=iso_code)
    .groupby(network.generators.carrier, axis=1)
    .sum()
    .resample(resample)
    .sum()
    .reset_index()
    .copy()
)

# get storage dispatch
storage_dispatch = (
    network
    .storage_units_t
    .p_dispatch
    .loc['2030-05-07 00:00:00':'2030-05-14 23:00:00']
    .filter(regex=iso_code)
    .resample(resample)
    .sum()
    .sum(axis=1)
    .to_numpy()
)

# append storage dispatch to generation
generation['battery'] = storage_dispatch

# get load
load = (
    network
    .loads_t
    .p
    .loc['2030-05-07 00:00:00':'2030-05-14 23:00:00']
    .filter(regex=iso_code)
    .sum(axis=1)
    .resample(resample)
    .sum()
    .reset_index()
)

# Create figure
fig = go.Figure()

# add traces
for generator in generation.columns:

    if generator != 'snapshot':
        fig.add_trace(
            go.Scatter(
                x=list(generation.snapshot),
                y=list(generation[generator].div(1e6)),
                mode='lines',
                stackgroup='one',
                name=generator,
                line=dict(color=network.carriers.color.to_dict()[generator], width=1),
            )
        )

# add load
fig.add_trace(
        go.Scatter(
        x=list(load.snapshot),
        y=list(load[0].div(1e6)),
        mode='lines',
        #stackgroup='one',
        name='Load',
        line=dict(color='black', width=1),
    )
)

# Add range slider
fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=5,
                        label="5D",
                        step="day",
                        stepmode="backward"),
                dict(count=1,
                        label="1M",
                        step="month",
                        stepmode="backward"),
                dict(count=3,
                        label="3M",
                        step="month",
                        stepmode="backward"),
                dict(count=1,
                        label="1Y",
                        step="year",
                        stepmode="backward"),
                # dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date",
    ),
    # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
    yaxis_title='Generation (TWh)',
    xaxis_title='Time (Hour)',
    title='Hourly Generation ' + iso_code + ' (2030-05-07 00:00:00 : 2030-05-14 23:00:00)',
    width=1200,
    height=500,
)
fig.show()

# fig.write_html('daily_generation_' + iso_code + '.html')
# fig.write_image('generation.png', width=1200, height=500, scale=2)

In [240]:
## DISPATCH AT THE LOWEST DEMAND WEEK

resample = 'H'
iso_code = 'MYSPE'

# get generation
generation = (
    network
    .generators_t
    .p
    .loc['2030-04-20 00:00:00':'2030-04-27 23:00:00']
    .filter(regex=iso_code)
    .groupby(network.generators.carrier, axis=1)
    .sum()
    .resample(resample)
    .sum()
    .reset_index()
    .copy()
)

# get storage dispatch
storage_dispatch = (
    network
    .storage_units_t
    .p_dispatch
    .loc['2030-04-20 00:00:00':'2030-04-27 23:00:00']
    .filter(regex=iso_code)
    .resample(resample)
    .sum()
    .sum(axis=1)
    .to_numpy()
)

# append storage dispatch to generation
generation['battery'] = storage_dispatch

# get load
load = (
    network
    .loads_t
    .p
    .loc['2030-04-20 00:00:00':'2030-04-27 23:00:00']
    .filter(regex=iso_code)
    .sum(axis=1)
    .resample(resample)
    .sum()
    .reset_index()
)

# Create figure
fig = go.Figure()

# add traces
for generator in generation.columns:

    if generator != 'snapshot':
        fig.add_trace(
            go.Scatter(
                x=list(generation.snapshot),
                y=list(generation[generator].div(1e6)),
                mode='lines',
                stackgroup='one',
                name=generator,
                line=dict(color=network.carriers.color.to_dict()[generator], width=1),
            )
        )

# add load
fig.add_trace(
        go.Scatter(
        x=list(load.snapshot),
        y=list(load[0].div(1e6)),
        mode='lines',
        #stackgroup='one',
        name='Load',
        line=dict(color='black', width=1),
    )
)

# Add range slider
fig.update_layout(
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=5,
                        label="5D",
                        step="day",
                        stepmode="backward"),
                dict(count=1,
                        label="1M",
                        step="month",
                        stepmode="backward"),
                dict(count=3,
                        label="3M",
                        step="month",
                        stepmode="backward"),
                dict(count=1,
                        label="1Y",
                        step="year",
                        stepmode="backward"),
                # dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date",
    ),
    # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
    yaxis_title='Generation (TWh)',
    xaxis_title='Time (Hour)',
    title='Hourly Generation ' + iso_code + ' (2030-04-20 00:00:00 : 2030-04-27 23:00:00)',
    width=1200,
    height=500,
)
fig.show()

# fig.write_html('daily_generation_' + iso_code + '.html')
# fig.write_image('generation.png', width=1200, height=500, scale=2)

In [ ]:
network.snapshots

In [ ]:
network.links_t.p0.sum().div(1e6)

In [ ]:
network.links.p_nom_opt

In [ ]:
network.storage_units_t.p_dispatch.sum()

In [ ]:
import tz_pypsa.wrangle as wrangle

run_name = 'SGPXX_fullCFE_20May_TP1_brownfield'

file_path = '/Users/yantidiah/Downloads/'+ run_name + '/solved_networks/'
file_destination = '/Users/yantidiah/Downloads/' + run_name + '/excels/'

filename_list = [
                #  'annual_matching_RES100_2030.nc',
                 'brownfield_2030.nc',
                #  'hourly_matching_CFE50_2030.nc',
                #  'hourly_matching_CFE60_2030.nc',
                #  'hourly_matching_CFE70_2030.nc',
                #  'hourly_matching_CFE80_2030.nc',
                #  'hourly_matching_CFE90_2030.nc',
                #  'hourly_matching_CFE95_2030.nc',
                #  'hourly_matching_CFE98_2030.nc',
                #  'hourly_matching_CFE99_2030.nc',
                #  'hourly_matching_CFE100_2030.nc',
                 ]

for name in filename_list:
    network = pypsa.Network()
    network.import_from_netcdf(file_path + name)
    wrangle.export_to_excel(network, file_destination + name.replace('.nc', '.xlsx'))
